# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [1]:
#1
class matrix:
    def __init__(self, rows, cols=None):
        if cols is not None:
            if rows <= 0 or cols <= 0:
                raise ValueError("Dimensions must be positive")

            self.data = []

            for i in range(rows):
                row = []
                for j in range(cols):
                    row.append(0.0)
                self.data.append(row)

        else:
            if not isinstance(rows, list) or len(rows) == 0:
                raise ValueError("Invalid matrix")

            if not isinstance(rows[0], list) or len(rows[0]) == 0:
                raise ValueError("Invalid matrix")

            for row in rows:
                if not isinstance(row, list) or len(row) != len(rows[0]):
                    raise ValueError("Rows must have equal length")

            self.data = []

            for row in rows:
                self.data.append(row[:])

    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]

        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        else:
            if len(value) != len(self.data[0]):
                raise ValueError("Wrong row size")

            self.data[key] = list(value)

    def copy_from(self, other):
        if isinstance(other, matrix):
            values = other.data
        else:
            values = other

        if len(values) != len(self.data):
            raise ValueError("Wrong number of rows")

        for i in range(len(values)):
            if len(values[i]) != len(self.data[0]):
                raise ValueError("Wrong number of columns")

        for i in range(len(values)):
            for j in range(len(values[0])):
                self.data[i][j] = values[i][j]


A = matrix(2, 3)
B = matrix([[1, 2, 3], [4, 5, 6]])

A[0][0] = 7
A[0, 1] = 8

print(A.data)
print(B[1][2])
print(B[1, 2])

A.copy_from(B)
print(A.data)

A.copy_from([[9, 8, 7], [6, 5, 4]])
print(A.data)

try:
    matrix([[1, 2], [3]])
except ValueError as error:
    print(error)

try:
    A.copy_from(matrix(1, 2))
except ValueError as error:
    print(error)

[[7, 8, 0.0], [0.0, 0.0, 0.0]]
6
6
[[1, 2, 3], [4, 5, 6]]
[[9, 8, 7], [6, 5, 4]]
Rows must have equal length
Wrong number of rows


In [2]:
#2
class matrix(matrix):
    def shape(self):
        return (len(self.data), len(self.data[0]))

    def transpose(self):
        n, m = self.shape()
        result = matrix(m, n)

        for i in range(n):
            for j in range(m):
                result[j][i] = self[i][j]

        return result

    def row(self, i):
        return matrix([self.data[i][:]])

    def column(self, j):
        result = []

        for row in self.data:
            result.append([row[j]])

        return matrix(result)

    def to_list(self):
        result = []

        for row in self.data:
            result.append(row[:])

        return result

    def block(self, n0, n1, m0, m1):
        return self[m0:m1, n0:n1]

    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key

            if isinstance(i, slice) or isinstance(j, slice):
                if isinstance(i, slice):
                    rows = self.data[i]
                else:
                    rows = [self.data[i]]

                result = []

                for row in rows:
                    if isinstance(j, slice):
                        part = row[j]
                    else:
                        part = [row[j]]

                    result.append(part)

                return matrix(result)

            return self.data[i][j]

        if isinstance(key, slice):
            return matrix(self.data[key])

        return self.data[key]


M = matrix([[1, 2, 3], [4, 5, 6]])

print(M.shape())
print(M.transpose().to_list())
print(M.row(1).to_list())
print(M.column(1).to_list())
print(M.to_list())
print(M.block(1, 3, 0, 2).to_list())
print(M[0:2, 1:3].to_list())

(2, 3)
[[1, 4], [2, 5], [3, 6]]
[[4, 5, 6]]
[[2], [5]]
[[1, 2, 3], [4, 5, 6]]
[[2, 3], [5, 6]]
[[2, 3], [5, 6]]


In [3]:
#3
def constant(n, m, c):
    result = matrix(n, m)

    for i in range(n):
        for j in range(m):
            result[i][j] = float(c)

    return result


def zeros(n, m):
    return constant(n, m, 0)


def ones(n, m):
    return constant(n, m, 1)


def eye(n):
    result = zeros(n, n)

    for i in range(n):
        result[i][i] = 1.0

    return result


A = constant(2, 3, 4)
B = zeros(2, 2)
C = ones(2, 2)
D = eye(3)

print(A.to_list())
print(B.to_list())
print(C.to_list())
print(D.to_list())

[[4.0, 4.0, 4.0], [4.0, 4.0, 4.0]]
[[0.0, 0.0], [0.0, 0.0]]
[[1.0, 1.0], [1.0, 1.0]]
[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]


In [4]:
#4
class matrix(matrix):
    def scalarmul(self, c):
        n, m = self.shape()
        result = matrix(n, m)

        for i in range(n):
            for j in range(m):
                result[i][j] = self[i][j] * c

        return result

    def add(self, N):
        if self.shape() != N.shape():
            raise ValueError("Sizes do not match")

        n, m = self.shape()
        result = matrix(n, m)

        for i in range(n):
            for j in range(m):
                result[i][j] = self[i][j] + N[i][j]

        return result

    def sub(self, N):
        if self.shape() != N.shape():
            raise ValueError("Sizes do not match")

        n, m = self.shape()
        result = matrix(n, m)

        for i in range(n):
            for j in range(m):
                result[i][j] = self[i][j] - N[i][j]

        return result

    def element_mult(self, N):
        if self.shape() != N.shape():
            raise ValueError("Sizes do not match")

        n, m = self.shape()
        result = matrix(n, m)

        for i in range(n):
            for j in range(m):
                result[i][j] = self[i][j] * N[i][j]

        return result

    def mat_mult(self, N):
        n, m = self.shape()
        p, q = N.shape()

        if m != p:
            raise ValueError("Sizes do not match")

        result = matrix(n, q)

        for i in range(n):
            for j in range(q):
                for k in range(m):
                    result[i][j] += self[i][k] * N[k][j]

        return result

    def equals(self, N):
        if not isinstance(N, matrix):
            return False

        return self.shape() == N.shape() and self.data == N.data


A = matrix([[1, 2], [3, 4]])
B = matrix([[2, 0], [1, 2]])

print(A.scalarmul(2).to_list())
print(A.add(B).to_list())
print(A.sub(B).to_list())
print(A.element_mult(B).to_list())
print(A.mat_mult(B).to_list())
print(A.equals(B))
print(A.equals(matrix([[1, 2], [3, 4]])))

try:
    A.add(matrix(3, 2))
except ValueError as error:
    print(error)

try:
    A.mat_mult(matrix(3, 2))
except ValueError as error:
    print(error)

[[2, 4], [6, 8]]
[[3, 2], [4, 6]]
[[-1, 2], [2, 2]]
[[2, 0], [3, 8]]
[[4.0, 4.0], [10.0, 8.0]]
False
True
Sizes do not match
Sizes do not match


In [5]:
#5
class matrix(matrix):
    def __add__(self, other):
        if not isinstance(other, matrix):
            return NotImplemented

        return self.add(other)

    def __sub__(self, other):
        if not isinstance(other, matrix):
            return NotImplemented

        return self.sub(other)

    def __mul__(self, other):
        if isinstance(other, matrix):
            return self.mat_mult(other)

        if isinstance(other, (int, float)):
            return self.scalarmul(other)

        return NotImplemented

    def __rmul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)

        return NotImplemented

    def __eq__(self, other):
        return self.equals(other)


M = matrix([[1, 2], [3, 4]])
N = matrix([[2, 0], [1, 2]])

print((2 * M).to_list())
print((M * 2).to_list())
print((M + N).to_list())
print((M - N).to_list())
print((M * N).to_list())
print(M == N)

[[2, 4], [6, 8]]
[[2, 4], [6, 8]]
[[3, 2], [4, 6]]
[[-1, 2], [2, 2]]
[[4.0, 4.0], [10.0, 8.0]]
False


In [6]:
#6
A = matrix([[1, 2], [3, 4]])
B = matrix([[2, 0], [1, 2]])
C = matrix([[1, 1], [0, 1]])
I = eye(2)

print((A * B) * C == A * (B * C))

print(A * (B + C) == A * B + A * C)

print(A * B != B * A)

print(A * I == A)

True
True
True
True
